# Análise individual: plano da China (`China_2025`)

**Projeto:** *As Relações Brasil-China e a Política de Governança Brasileira sobre Inteligência Artificial* (FAPESP, Processo nº 2026/04597-1)
**Etapa:** 02 (análise individual), conforme a skill 02 ([`skill_análise_individual.md`](skill_análise_individual.md))
**Insumo:** `IA_Plus_2025_extracao.json`, produzido pela skill 01 ([`skill_extração.md`](skill_extração.md))

---

> ⚠️ **Visualizações prévias.** Todas as visualizações geradas neste notebook (gráficos e tabelas) são **prévias** e compõem **apenas uma base para o início da construção de novos gráficos**. As novas visualizações serão construídas de acordo com os comandos e as exigências do pesquisador, que são obedecidos em última instância. Nenhuma visualização, tabela ou relatório vai para a pasta `resultados/` sem comando expresso do pesquisador.

## 1. Do que se trata

Este notebook analisa, de forma **individual**, o texto do plano da China *Opinions of the State Council on Deepening the Implementation of the “Artificial Intelligence+” Initiative*. Ele faz parte do fluxo de análise da pasta *Planos Nacionais de IA*: a skill 01 extrai o texto essencial de cada plano; esta etapa (skill 02) analisa cada plano isoladamente; a skill 03 compara os planos entre si.

## 2. Finalidade

A finalidade deste notebook é **gerar visualizações e interpretações sobre o plano analisado**, por meio de gráficos, tabelas e medidas textuais. As análises permitem conhecer:

- a **extensão** e a estrutura do texto;
- os **termos mais comuns** e como se distribuem ao longo do documento;
- as **expressões recorrentes**.

## 3. Insumo e recorte do conteúdo

São utilizados **apenas os resultados da skill 01**, ou seja, o arquivo JSON do plano gerado por aquela etapa. Desse arquivo, interessam **somente**:

- o **título** e o **subtítulo** do plano (`document.title` e `document.subtitle`), que identificam o documento e, por fazerem parte do texto essencial, **entram na contagem** como unidade 0 (`INCLUIR_TITULO_NA_CONTAGEM = True`). Quando não há subtítulo, a unidade 0 é só o título;
- o **conteúdo integral e literal de cada unidade** (`sections[].text`): títulos internos, parágrafos e itens de lista. É o único conteúdo analisado;
- o **tipo** (`type`) e a **categoria estrutural** (`category`) de cada unidade, usados apenas como **filtros**. Por padrão, os títulos internos entram (`INCLUIR_TITULOS_INTERNOS = True`) e nenhuma categoria é retirada (`CATEGORIAS_EXCLUIDAS = []`).

O identificador de cada unidade (`id`) é usado apenas como índice de posição no documento, e o bloco `extraction`, apenas para verificar a integridade do arquivo. O caminho de títulos (`path`) só serve para agrupar unidades por parte ou eixo quando o pesquisador pedir, e não é usado neste notebook. **Todo conteúdo periférico fica fora da análise**: numeração (`number`), nível (`level`), página (`page`) e demais metadados (`issuer`, `date`, `period` etc.). A função `carregar_extracao` descarta esse conteúdo no momento da leitura. O relatório de extração da skill 01 é apenas informativo e não é consultado.

## 4. Método (visão geral)

| Etapa | Procedimento |
|---|---|
| Verificação | Integridade do JSON e origem na skill 01 para o corpus *Planos Nacionais de IA* |
| Recorte | Título (unidade 0), títulos internos, parágrafos e itens; cada filtro é registrado no perfil do documento, com as unidades e os tokens que retirou |
| Pré-processamento | spaCy (`en_core_web_sm`): tokenização, reunião de compostos com hífen, lematização, classes gramaticais, *stopwords* |
| Frequências | Frequência absoluta, por 1.000 tokens, alcance (unidades em que o termo aparece) e dispersão (DP normalizado de Gries) |
| Expressões | Bigramas de tokens de conteúdo adjacentes no texto original |

O detalhamento de cada procedimento está na configuração, nas funções comentadas e no texto que acompanha cada visualização.

## 5. Transparência das escolhas

**Todo método utilizado e todas as escolhas tomadas estão presentes e verificáveis** em dois lugares:

1. **no código**: os parâmetros ficam reunidos na célula de configuração, e cada função traz comentários sobre o que faz;
2. **na descrição que segue cada visualização**, composta por: *Método e escolhas*, *Como ler*, *Interpretação* e *Fragilidades e ajustes possíveis*.

As escolhas do pesquisador prevalecem sobre qualquer padrão deste notebook. As decisões tomadas pelo agente dentro do que o pesquisador não definiu (por exemplo, os ajustes de tokenização da seção 10) são reversíveis pela alteração de um parâmetro da configuração.

As tabelas completas (e não apenas os recortes exibidos nos gráficos), as figuras e os demais arquivos **só são exportados para a pasta `resultados/` por comando expresso do pesquisador** (skill 02, seção 4.4). Sem esse comando, o notebook apenas exibe os resultados.

## 6. Princípios

Todo o conteúdo deste notebook segue os princípios da **skill 02** (análise individual dos planos). Por isso, **não incorpora análise conjunta**: nenhum resultado aqui é comparado com outro plano. A comparação entre planos é feita em notebook próprio, segundo a skill 03.

> As interpretações foram redigidas **somente após a execução** do notebook, com base nos valores efetivamente obtidos com o JSON de SHA-256 `db4c5ba3e43a089646f23cae3dcaddce6aca4190166d40f7925794e749808fd2` (exibido na seção 9) e com a configuração da seção 7. Cada interpretação separa **observação** (o que os números mostram) de **inferência** (o que eles podem sugerir). Se o JSON ou a configuração mudarem, as interpretações devem ser revistas.

## 7. Configuração

Todos os parâmetros editáveis estão nesta célula. Qualquer alteração deve ser registrada. A gravação em `resultados/` está desativada (`SALVAR_SAIDAS = False`) e só é ativada por comando do pesquisador.

In [ ]:
import hashlib
import json
import math
import platform
import re
import unicodedata
import zlib
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
from IPython.display import Markdown, display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter, MaxNLocator
from spacy.util import compile_suffix_regex

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 140)

# =====================================================================
# CONFIGURAÇÃO — parâmetros editáveis pelo pesquisador
# =====================================================================
SIGLA = "China_2025"
ARQUIVO_JSON = "IA_Plus_2025_extracao.json"     # entregável da skill 01
PAIS_OU_BLOCO = "China"                         # define a cor fixa do plano (COR_DOCUMENTO, seção 8)
ARQUIVO_CODEBOOK = None                         # categorias temáticas: só por comando do pesquisador

# Saídas (figuras, tabelas, registro). Resultados anteriores nunca são sobrescritos:
# para uma nova execução com mudanças, altere VERSAO_SAIDA (v2, v3, ...).
NOME_SAIDA = SIGLA
VERSAO_SAIDA = "v1"
# Gravação em resultados/ DESATIVADA: visualizações, tabelas e relatórios só vão para
# resultados/ por comando expresso do pesquisador (skill, seção 4.4).
SALVAR_SAIDAS = False
SOBRESCREVER = False

# Recorte do conteúdo (skill, seção 3.1). Todo filtro é informado no perfil do documento.
INCLUIR_TITULO_NA_CONTAGEM = True   # título e subtítulo entram na contagem como unidade 0
INCLUIR_TITULOS_INTERNOS = True     # unidades do tipo "heading"
CATEGORIAS_EXCLUIDAS = []           # ex.: ["Foreword", "Executive summary", "Box", "Annex"] (decisão do pesquisador)
UNIDADES_EXCLUIDAS = {}             # {id: "motivo"}: tratamento de inflação (skill, seção 3.5)

# Pré-processamento
IDIOMA_ESPERADO = "en"
MODELO_SPACY = "en_core_web_sm"
CLASSES_DE_CONTEUDO = {"NOUN", "PROPN", "VERB", "ADJ", "ADV"}
TAMANHO_MINIMO_LEMA = 2
STOPWORDS_ADICIONAIS = []    # lemas a remover além da lista do spaCy (decisão do pesquisador)
STOPWORDS_MANTIDAS = []      # lemas da lista do spaCy que devem ser mantidos (decisão do pesquisador)
CORRECOES_DE_LEMA = {
    "datum": "data",         # o spaCy lematiza "data" como "datum"; mantém-se "data", forma usada nos textos
}

# Ajustes de tokenização (explicados na seção 10; ambos reversíveis aqui)
SEPARAR_SIMBOLO_FINAL = True     # separa o "+" (AI+) e o hífen suspenso (intelligentization- and) colados
                                 # à palavra: a palavra é contada e o símbolo não
SIGLAS_COM_E_COMERCIAL = True    # siglas com "&" (R&D, S&T) contam como um token, como as siglas com pontos (U.S.)

# Exame de inflação (skill, seção 3.5)
TAMANHO_MINIMO_FRASE_REPETIDA = 8    # frases idênticas com pelo menos N tokens são sinalizadas

# Recortes de visualização (as tabelas exportadas são sempre completas)
TOP_N_LEMAS = 25
TOP_N_DISPERSAO = 15
TOP_N_BIGRAMAS = 20
FREQ_MINIMA_BIGRAMA = 2

# Incerteza (bootstrap por unidades)
N_BOOTSTRAP = 2000
NIVEL_CONFIANCA = 0.95
SEMENTE = 2026


## 8. Ambiente, estilo e funções

Restrição de pasta, pasta de saída (usada apenas quando o pesquisador ordenar a gravação em `resultados/`), padrão gráfico e funções auxiliares.

In [ ]:
# ---------------------------------------------------------------------------
# Restrição de ambiente: o notebook só roda de dentro da pasta "Planos Nacionais de IA".
# ---------------------------------------------------------------------------
PASTA = Path.cwd()
if unicodedata.normalize("NFC", PASTA.name) != "Planos Nacionais de IA":
    raise RuntimeError(
        "Este notebook deve ser executado de dentro da pasta 'Planos Nacionais de IA' "
        f"(restrição de ambiente da skill). Pasta atual: {PASTA}"
    )

# ---------------------------------------------------------------------------
# Pasta de saída versionada: resultados anteriores nunca são sobrescritos.
# ---------------------------------------------------------------------------
DIR_SAIDA = PASTA / "resultados" / NOME_SAIDA / VERSAO_SAIDA
if SALVAR_SAIDAS:
    if DIR_SAIDA.exists() and any(DIR_SAIDA.iterdir()) and not SOBRESCREVER:
        raise FileExistsError(
            f"A pasta de saída '{DIR_SAIDA.relative_to(PASTA)}' já contém arquivos. "
            "Para preservar a versão anterior, altere VERSAO_SAIDA (ex.: 'v2') na configuração. "
            "Use SOBRESCREVER = True somente se a substituição for intencional."
        )
    DIR_SAIDA.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Estilo das figuras (padrão acadêmico: fonte >= 9 pt, 300 dpi, PNG + PDF)
# ---------------------------------------------------------------------------
# Cores fixas por país ou bloco, iguais em todos os notebooks da pasta (skill, seção 3.7).
# Paleta inicial, editável pelo pesquisador. Planos do mesmo país ou bloco recebem tons da mesma cor.
COR_DOCUMENTO = {
    "Brasil": "#2E9E4F",           # verde
    "Estados Unidos": "#1F3F8F",   # azul-escuro
    "União Europeia": "#EDB120",   # amarelo
    "China": "#B2182B",            # vermelho
}
COR_TINTA = "#0b0b0b"      # texto principal
COR_TINTA_2 = "#52514e"    # texto secundário (rótulos, anotações)
COR_GRADE = "#e4e3df"      # grade e linhas de apoio
COR_LIGACAO = "#b9b8b2"    # segmentos de ligação (gráficos de halteres)
COR_ESTRUTURA = "#8f8e88"  # cinza neutro: título e títulos internos no gráfico de extensão

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": COR_TINTA_2,
    "axes.linewidth": 0.6,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": COR_GRADE,
    "grid.linewidth": 0.6,
    "grid.linestyle": "-",
    "text.color": COR_TINTA,
    "axes.labelcolor": COR_TINTA,
    "xtick.color": COR_TINTA_2,
    "ytick.color": COR_TINTA_2,
    "legend.frameon": False,
    "pdf.fonttype": 42,
})

# ---------------------------------------------------------------------------
# Funções auxiliares de formatação e gravação
# ---------------------------------------------------------------------------
def num(x, casas=2):
    """Formata número no padrão brasileiro (vírgula decimal, ponto de milhar)."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "—"
    texto = f"{x:,.{casas}f}"
    return texto.replace(",", "§").replace(".", ",").replace("§", ".")


# Rótulos de eixo com vírgula decimal.
FMT_BR = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))


def exibir(df, casas=2):
    """Exibe uma tabela com vírgula decimal, sem alterar os valores."""
    display(df.style.format(precision=casas, decimal=",", thousands=".", na_rep="—"))


def salvar_figura(fig, nome):
    """Grava a figura em PNG (300 dpi) e PDF na pasta de saída versionada."""
    if SALVAR_SAIDAS:
        for ext in ("png", "pdf"):
            fig.savefig(DIR_SAIDA / f"{nome}.{ext}")


def salvar_tabela(df, nome, index=True):
    """Grava a tabela completa em CSV (UTF-8 com BOM, ';', vírgula decimal)."""
    if SALVAR_SAIDAS:
        df.to_csv(DIR_SAIDA / f"{nome}.csv", sep=";", decimal=",", encoding="utf-8-sig", index=index)


def semente_documento(sigla):
    """Semente do bootstrap própria de cada documento, derivada de SEMENTE e da sigla.
    Garante (a) os mesmos intervalos no notebook individual e no comparativo e
    (b) sequências aleatórias independentes entre documentos diferentes."""
    return np.random.SeedSequence([SEMENTE, zlib.crc32(sigla.encode("utf-8"))])


print(f"Pasta de trabalho: {PASTA}")
print(f"Saídas: {DIR_SAIDA.relative_to(PASTA) if SALVAR_SAIDAS else 'gravação desativada'}")

if PAIS_OU_BLOCO not in COR_DOCUMENTO:
    raise KeyError(f"PAIS_OU_BLOCO = {PAIS_OU_BLOCO!r} não está na paleta COR_DOCUMENTO: {list(COR_DOCUMENTO)}")
COR = COR_DOCUMENTO[PAIS_OU_BLOCO]


In [ ]:
# ===========================================================================
# PIPELINE DE PRÉ-PROCESSAMENTO E MEDIDAS
# (idêntico em todos os notebooks de análise da pasta "Planos Nacionais de IA")
# ===========================================================================

# Token (palavra) = sequência de letras, admitindo hífen interno (multi-stakeholder),
# sigla com pontos (U.S.) e, se SIGLAS_COM_E_COMERCIAL = True, sigla com "&" (R&D).
# Números, pontuação e símbolos NÃO são tokens.
RE_PALAVRA = re.compile(
    r"(?:[^\W\d_]\.){2,}"
    + (r"|[^\W\d_]+(?:&[^\W\d_]+)+" if SIGLAS_COM_E_COMERCIAL else "")
    + r"|[^\W\d_]+(?:[-‐‑][^\W\d_]+)*"
)
HIFENS = {"-", "‐", "‑"}

# Tipos de unidade (a unidade 0 reúne o título e o subtítulo).
ROTULO_TIPO = {
    "title": "título e subtítulo (id 0)",
    "heading": "títulos internos",
    "paragraph": "parágrafos",
    "list_item": "itens de lista",
}


def normalizar_hifen(texto):
    """Uniformiza variantes tipográficas do hífen (apenas nas chaves de contagem;
    o texto do JSON não é alterado)."""
    return texto.replace("‐", "-").replace("‑", "-")


def carregar_extracao(caminho):
    """Lê o JSON produzido pela skill 01 e devolve SOMENTE o título, o subtítulo e as
    unidades (id, tipo, categoria, caminho de títulos e texto integral). Todo o conteúdo
    periférico (numeração, nível, página e demais metadados) é descartado aqui.
    Interrompe a execução se o arquivo não passar nas verificações de integridade."""
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo '{caminho.name}' não encontrado na pasta. "
            "Execute antes a skill 01 (skill_extração.md) para gerar o JSON do plano."
        )
    bruto = caminho.read_bytes()
    dados = json.loads(bruto.decode("utf-8"))

    problemas = [f"chave '{c}' ausente" for c in ("document", "extraction", "sections") if c not in dados]
    if problemas:
        raise ValueError("Falha na verificação do JSON:\n- " + "\n- ".join(problemas))

    extracao = dados["extraction"]
    if extracao.get("skill") != "skill_extração":
        problemas.append("o JSON não registra ter sido produzido pela skill 01 (extraction.skill)")
    if extracao.get("corpus") != "Planos Nacionais de IA":
        problemas.append("o JSON não registra o corpus 'Planos Nacionais de IA' (extraction.corpus)")
    secoes = dados["sections"]
    if [s.get("id") for s in secoes] != list(range(1, len(secoes) + 1)):
        problemas.append("os ids das unidades não são sequenciais a partir de 1")
    vazias = [s.get("id") for s in secoes if not isinstance(s.get("text"), str) or not s["text"].strip()]
    if vazias:
        problemas.append(f"unidades com texto vazio: {vazias}")
    tipos_invalidos = [s.get("id") for s in secoes if s.get("type") not in ("heading", "paragraph", "list_item")]
    if tipos_invalidos:
        problemas.append(f"unidades com tipo fora de heading/paragraph/list_item: {tipos_invalidos}")
    # Contadores registrados pela skill 01 x conteúdo efetivo do JSON
    efetivos = {
        "units_extracted": len(secoes),
        "headings_extracted": sum(s.get("type") == "heading" for s in secoes),
        "paragraphs_extracted": sum(s.get("type") == "paragraph" for s in secoes),
        "list_items_extracted": sum(s.get("type") == "list_item" for s in secoes),
    }
    for chave, valor in efetivos.items():
        if extracao.get(chave) is not None and extracao[chave] != valor:
            problemas.append(f"extraction.{chave} = {extracao[chave]}, mas o JSON tem {valor}")
    if not dados["document"].get("title"):
        problemas.append("título (document.title) ausente")
    if problemas:
        raise ValueError("Falha na verificação do JSON:\n- " + "\n- ".join(problemas))

    return {
        "arquivo": caminho.name,
        "sha256": hashlib.sha256(bruto).hexdigest(),
        "titulo": dados["document"]["title"],
        "subtitulo": dados["document"].get("subtitle"),
        "idioma": dados["document"].get("language"),
        # SOMENTE id (posição), tipo e categoria (filtros), path (agrupador estrutural,
        # usado só por comando do pesquisador) e texto (único conteúdo analisado)
        "unidades": [
            {"id": s["id"], "tipo": s["type"], "categoria": s.get("category"),
             "path": s.get("path") or [], "texto": s["text"]}
            for s in secoes
        ],
    }


def selecionar_unidades(documento):
    """Monta as unidades na ordem do documento (unidade 0 = título e subtítulo) e aplica
    os filtros da configuração (skill, seção 3.1). Devolve (selecionadas, retiradas):
    listas de dicionários {id, tipo, categoria, texto, filtro}; nas retiradas, 'filtro'
    é o parâmetro responsável pela retirada."""
    ids_json = {u["id"] for u in documento["unidades"]}
    desconhecidos = sorted(set(UNIDADES_EXCLUIDAS) - ids_json)
    if desconhecidos:
        raise ValueError(f"UNIDADES_EXCLUIDAS contém ids que não existem no JSON: {desconhecidos}")
    # A quebra de linha entre título e subtítulo impede que a última palavra de um e a
    # primeira do outro sejam tratadas como adjacentes (bigramas).
    texto_0 = "\n".join(t for t in (documento["titulo"], documento["subtitulo"]) if t)
    candidatas = [{"id": 0, "tipo": "title", "categoria": None, "texto": texto_0}]
    candidatas += [{k: u[k] for k in ("id", "tipo", "categoria", "texto")} for u in documento["unidades"]]
    selecionadas, retiradas = [], []
    for u in candidatas:
        if u["tipo"] == "title" and not INCLUIR_TITULO_NA_CONTAGEM:
            filtro = "INCLUIR_TITULO_NA_CONTAGEM"
        elif u["tipo"] == "heading" and not INCLUIR_TITULOS_INTERNOS:
            filtro = "INCLUIR_TITULOS_INTERNOS"
        elif u["categoria"] in CATEGORIAS_EXCLUIDAS:
            filtro = "CATEGORIAS_EXCLUIDAS"
        elif u["id"] in UNIDADES_EXCLUIDAS:
            filtro = "UNIDADES_EXCLUIDAS"
        else:
            filtro = None
        (retiradas if filtro else selecionadas).append({**u, "filtro": filtro})
    if not selecionadas:
        raise ValueError("Os filtros da configuração retiraram todas as unidades.")
    return selecionadas, retiradas


def tabela_filtros(retiradas, tokens_retirados):
    """Unidades e tokens retirados por cada filtro da configuração (skill, seção 3.1)."""
    filtros = {
        "INCLUIR_TITULO_NA_CONTAGEM": ("Título e subtítulo (unidade 0)", INCLUIR_TITULO_NA_CONTAGEM),
        "INCLUIR_TITULOS_INTERNOS": ("Títulos internos", INCLUIR_TITULOS_INTERNOS),
        "CATEGORIAS_EXCLUIDAS": ("Categorias estruturais", CATEGORIAS_EXCLUIDAS),
        "UNIDADES_EXCLUIDAS": ("Unidades específicas (tratamento de inflação)", UNIDADES_EXCLUIDAS),
    }
    tam = tokens_retirados.groupby("unidade").size()
    linhas = []
    for parametro, (nome, valor) in filtros.items():
        ids = [u["id"] for u in retiradas if u["filtro"] == parametro]
        linhas.append({
            "filtro": nome,
            "parametro": f"{parametro} = {valor!r}",
            "unidades_retiradas": len(ids),
            "tokens_retirados": int(tam.reindex(ids, fill_value=0).sum()),
            "ids_retirados": ", ".join(map(str, ids)) if ids else "—",
        })
    return pd.DataFrame(linhas).set_index("filtro")


def ajustar_tokenizador(nlp):
    """Se SEPARAR_SIMBOLO_FINAL = True, separa do fim da palavra o "+" (AI+) e o hífen
    suspenso ("intelligentization- and"), que o tokenizador do spaCy mantém colados.
    Sem o ajuste, a palavra inteira deixa de ser token, porque o símbolo impede o seu
    reconhecimento como palavra; com ele, a palavra é contada e o símbolo não."""
    if SEPARAR_SIMBOLO_FINAL:
        sufixos = list(nlp.Defaults.suffixes) + [r"(?<=[^\W\d_])\+", r"(?<=[^\W\d_])[-‐‑]"]
        nlp.tokenizer.suffix_search = compile_suffix_regex(sufixos).search
    return nlp


def tabela_ajustes_tokenizacao(unidades):
    """Formas do texto afetadas pelos ajustes de tokenização (SEPARAR_SIMBOLO_FINAL e
    SIGLAS_COM_E_COMERCIAL), com o número de ocorrências nas unidades analisadas e o
    tratamento aplicado. A contagem é feita diretamente no texto das unidades."""
    regras = [
        ("'+' colado ao fim da palavra", r"[^\W\d_]+\+(?![^\W\d_])", SEPARAR_SIMBOLO_FINAL,
         "palavra contada; o '+' (símbolo) não é token"),
        ("hífen suspenso", r"[^\W\d_]+[-‐‑](?=\s)", SEPARAR_SIMBOLO_FINAL,
         "palavra contada; o hífen (pontuação) não é token"),
        ("sigla com '&'", r"(?<![^\W\d_])[^\W\d_]+(?:&[^\W\d_]+)+(?![^\W\d_])", SIGLAS_COM_E_COMERCIAL,
         "sigla contada como um token"),
    ]
    linhas = []
    for ajuste, padrao, ativo, tratamento in regras:
        contagem = Counter(m.group(0) for _, texto in unidades for m in re.finditer(padrao, texto))
        for forma, n in sorted(contagem.items(), key=lambda kv: (-kv[1], kv[0])):
            linhas.append({
                "ajuste": ajuste,
                "forma_no_texto": forma,
                "ocorrencias": n,
                "tratamento": tratamento if ativo else "ajuste desativado: a forma inteira não é contada como token",
            })
    return pd.DataFrame(linhas, columns=["ajuste", "forma_no_texto", "ocorrencias", "tratamento"])


def montar_stopwords(nlp):
    """Lista de stopwords do spaCy, ajustada pelas listas editáveis da configuração."""
    base = {w.lower() for w in nlp.Defaults.stop_words}
    return (base | {w.lower() for w in STOPWORDS_ADICIONAIS}) - {w.lower() for w in STOPWORDS_MANTIDAS}


def unir_compostos_hifenizados(doc):
    """Reúne em um único token as palavras compostas com hífen (ex.: multi-stakeholder,
    human-centric, state-of-the-art), que o spaCy separa em vários tokens.
    Só une quando não há espaço em volta do hífen."""
    trechos, i, n = [], 0, len(doc)
    while i < n:
        j = i
        while (j + 2 < n and doc[j].is_alpha and doc[j].whitespace_ == ""
               and doc[j + 1].text in HIFENS and doc[j + 1].whitespace_ == ""
               and doc[j + 2].is_alpha):
            j += 2
        if j > i:
            trechos.append(doc[i:j + 1])
            i = j + 1
        else:
            i += 1
    with doc.retokenize() as rt:
        for trecho in trechos:
            ultimo = trecho[-1]
            # Lema do composto = forma escrita; se o último elemento estiver no plural,
            # usa-se o lema dele (multi-stakeholders -> multi-stakeholder).
            if ultimo.tag_ in {"NNS", "NNPS"}:
                lema = trecho.text[: len(trecho.text) - len(ultimo.text)] + ultimo.lemma_
            else:
                lema = trecho.text
            rt.merge(trecho, attrs={"LEMMA": lema, "POS": ultimo.pos_, "TAG": ultimo.tag_})
    return doc


def processar_documento(unidades, nlp, stopwords):
    """Aplica o pré-processamento a cada unidade (título, título interno, parágrafo ou item)
    e devolve uma tabela com uma linha por token (palavra), na ordem original do documento."""
    linhas, posicao = [], 0
    for (uid, _), doc in zip(unidades, nlp.pipe([texto for _, texto in unidades])):
        doc = unir_compostos_hifenizados(doc)
        for k, frase in enumerate(doc.sents):
            for tok in frase:
                if not RE_PALAVRA.fullmatch(tok.text):
                    continue
                lema = normalizar_hifen(tok.lemma_.lower())
                lema = CORRECOES_DE_LEMA.get(lema, lema)
                linhas.append({
                    "unidade": uid,
                    "frase": f"{uid}.{k}",
                    "i": tok.i,                      # índice na unidade (detecta adjacência)
                    "posicao": posicao,              # posição no documento inteiro
                    "forma": normalizar_hifen(tok.text.lower()),
                    "lema": lema,
                    "classe": tok.pos_,
                    "etiqueta": tok.tag_,
                })
                posicao += 1
    tokens = pd.DataFrame(linhas, columns=["unidade", "frase", "i", "posicao", "forma", "lema", "classe", "etiqueta"])
    tokens["conteudo"] = (
        tokens["classe"].isin(CLASSES_DE_CONTEUDO)
        & ~tokens["lema"].isin(stopwords)
        & (tokens["lema"].str.len() >= TAMANHO_MINIMO_LEMA)
        & tokens["lema"].map(lambda lema: bool(RE_PALAVRA.fullmatch(lema)))
    ).astype(bool)
    return tokens


def perfil_documento(tokens, tipos, rotulo):
    """Indicadores descritivos de extensão e vocabulário. tipos: {id da unidade: tipo}."""
    conteudo = tokens[tokens["conteudo"]]
    tam = tokens.groupby("unidade").size()
    freq_cont = conteudo["lema"].value_counts()
    por_tipo = pd.Series(tipos).value_counts()
    indicadores = {"Unidades analisadas": len(tipos)}
    for tipo, nome in ROTULO_TIPO.items():
        indicadores[f"Unidades: {nome}"] = int(por_tipo.get(tipo, 0))
    indicadores.update({
        "Unidades sem nenhum token": len(tipos) - tokens["unidade"].nunique(),
        "Tokens (palavras)": len(tokens),
        "Formas distintas": tokens["forma"].nunique(),
        "Tokens de conteúdo": len(conteudo),
        "Tokens de conteúdo (% dos tokens)": len(conteudo) / len(tokens) * 100,
        "Lemas de conteúdo distintos": conteudo["lema"].nunique(),
        "Lemas de conteúdo com uma única ocorrência": int((freq_cont == 1).sum()),
        "Tokens por unidade: média": tam.mean(),
        "Tokens por unidade: mediana": tam.median(),
        "Tokens por unidade: mínimo": tam.min(),
        "Tokens por unidade: máximo": tam.max(),
    })
    return pd.Series(indicadores, name=rotulo)


def tabela_stopwords_descartadas(tokens, stopwords):
    """Lemas de classes de conteúdo (CLASSES_DE_CONTEUDO) que a lista de stopwords retirou
    da contagem de conteúdo (skill, seção 4.2), com frequência, classes e formas observadas.
    Ordenação: frequência decrescente; empate -> ordem alfabética."""
    sub = tokens[tokens["classe"].isin(CLASSES_DE_CONTEUDO) & tokens["lema"].isin(stopwords)]
    if sub.empty:
        return pd.DataFrame(columns=["frequencia", "classes", "formas_observadas"]).rename_axis("lema")
    grupos = sub.groupby("lema")
    tab = pd.DataFrame({
        "frequencia": grupos.size(),
        "classes": grupos["classe"].agg(lambda s: "; ".join(f"{c} ({n})" for c, n in s.value_counts().items())),
        "formas_observadas": grupos["forma"].agg(
            lambda s: "; ".join(f"{f} ({n})" for f, n in s.value_counts().items())
        ),
    })
    tab.index.name = "lema"
    tab = tab.reset_index().sort_values(["frequencia", "lema"], ascending=[False, True])
    return tab.set_index("lema")


def exame_de_repeticoes(unidades, tokens):
    """Exame de inflação (skill, seção 3.5): unidades com texto idêntico e frases idênticas,
    com pelo menos TAMANHO_MINIMO_FRASE_REPETIDA tokens, repetidas no documento.
    Devolve duas tabelas, vazias quando não há repetição."""
    textos = pd.Series({uid: " ".join(texto.split()).casefold() for uid, texto in unidades}, name="texto")
    unidades_rep = textos[textos.duplicated(keep=False)].rename_axis("unidade").to_frame()
    frases = tokens.groupby("frase", sort=False).agg(
        unidade=("unidade", "first"), n_tokens=("forma", "size"), texto=("forma", " ".join))
    frases = frases[frases["n_tokens"] >= TAMANHO_MINIMO_FRASE_REPETIDA]
    frases_rep = frases[frases["texto"].duplicated(keep=False)]
    return unidades_rep, frases_rep


def tabela_frequencias(tokens):
    """Frequência, taxa por 1.000 tokens, alcance e dispersão (DP normalizado) de
    TODOS os lemas de conteúdo. Ordenação: frequência decrescente; empate -> ordem alfabética."""
    n_tokens = len(tokens)
    unidades = sorted(tokens["unidade"].unique())
    proporcao_unidade = (tokens.groupby("unidade").size().reindex(unidades) / n_tokens).to_numpy()
    conteudo = tokens[tokens["conteudo"]]
    cruz = pd.crosstab(conteudo["lema"], conteudo["unidade"]).reindex(columns=unidades, fill_value=0)
    freq = cruz.sum(axis=1)
    # DP de Gries (2008), normalizado por Lijffijt & Gries (2012):
    # 0 = distribuição proporcional ao tamanho das unidades; 1 = concentração máxima.
    v = cruz.div(freq, axis=0).to_numpy()
    dp = 0.5 * np.abs(v - proporcao_unidade).sum(axis=1)
    dp_norm = dp / (1 - proporcao_unidade.min()) if len(unidades) > 1 else np.full(len(freq), np.nan)
    formas = conteudo.groupby("lema")["forma"].agg(
        lambda s: "; ".join(f"{forma} ({n})" for forma, n in s.value_counts().items())
    )
    tab = pd.DataFrame({
        "frequencia": freq,
        "por_1000_tokens": freq / n_tokens * 1000,
        "unidades_com_ocorrencia": (cruz > 0).sum(axis=1),
        "pct_unidades": (cruz > 0).sum(axis=1) / len(unidades) * 100,
        "DP_norm": dp_norm,
        "formas_observadas": formas,
    })
    tab.index.name = "lema"
    tab = tab.reset_index().sort_values(["frequencia", "lema"], ascending=[False, True])
    return tab.set_index("lema")


def relatar_empates(tab, coluna, n, rotulo):
    """Informa termos empatados no ponto de corte de um recorte 'top N'."""
    if len(tab) <= n:
        return
    corte = tab[coluna].iloc[n - 1]
    fora = tab.iloc[n:]
    fora = fora[fora[coluna] == corte]
    if len(fora):
        lista = ", ".join(fora.index[:20]) + (" …" if len(fora) > 20 else "")
        print(
            f"Empate no ponto de corte: o {n}º {rotulo} tem valor {num(corte, 2)}; "
            f"outros {len(fora)} com o mesmo valor ficaram fora do recorte "
            f"(desempate por ordem alfabética): {lista}"
        )


def tabela_bigramas(tokens):
    """Pares de tokens de conteúdo ADJACENTES no texto original (mesma frase, sem
    pontuação entre eles). Não se formam pares artificiais pela remoção de stopwords."""
    a = tokens.iloc[:-1].reset_index(drop=True)
    b = tokens.iloc[1:].reset_index(drop=True)
    ok = (
        (a["frase"].to_numpy() == b["frase"].to_numpy())
        & (b["i"].to_numpy() - a["i"].to_numpy() == 1)
        & a["conteudo"].to_numpy() & b["conteudo"].to_numpy()
    )
    pares = pd.DataFrame({
        "bigrama": (a["lema"] + " " + b["lema"])[ok],
        "forma": (a["forma"] + " " + b["forma"])[ok],
        "unidade": a["unidade"][ok],
    })
    if pares.empty:
        return pd.DataFrame(columns=["frequencia", "por_1000_tokens", "unidades_com_ocorrencia",
                                     "forma_principal", "formas_observadas"]).rename_axis("bigrama")
    grupos = pares.groupby("bigrama")
    tab = pd.DataFrame({
        "frequencia": grupos.size(),
        "por_1000_tokens": grupos.size() / len(tokens) * 1000,
        "unidades_com_ocorrencia": grupos["unidade"].nunique(),
        "forma_principal": grupos["forma"].agg(lambda s: s.value_counts().index[0]),
        "formas_observadas": grupos["forma"].agg(
            lambda s: "; ".join(f"{f} ({n})" for f, n in s.value_counts().items())
        ),
    })
    tab = tab.reset_index().sort_values(["frequencia", "bigrama"], ascending=[False, True])
    return tab.set_index("bigrama")


def carregar_codebook(caminho):
    """Lê o codebook e verifica a exclusividade mútua das categorias.
    Devolve o codebook e a lista de entradas ordenada da expressão mais longa para a mais curta."""
    codebook = json.loads(caminho.read_text(encoding="utf-8"))
    entradas, vistos = [], {}
    for categoria, info in codebook["categorias"].items():
        for termo in info["termos"]:
            chave = tuple(normalizar_hifen(termo.lower()).split())
            if chave in vistos:
                raise ValueError(
                    f"Termo '{termo}' aparece em '{vistos[chave]}' e em '{categoria}'. "
                    "As categorias do codebook devem ser mutuamente exclusivas."
                )
            vistos[chave] = categoria
            entradas.append((chave, categoria, termo))
    entradas.sort(key=lambda e: -len(e[0]))
    sha = hashlib.sha256(caminho.read_bytes()).hexdigest()
    return codebook, entradas, sha


def aplicar_codebook(tokens, entradas):
    """Localiza os termos do codebook no texto. Cada palavra do termo é comparada com a
    forma OU com o lema do token. Expressões exigem palavras contíguas na mesma frase.
    Prevalece a expressão mais longa, e cada token é contado no máximo uma vez."""
    achados = []
    for _, frase in tokens.groupby("frase", sort=False):
        formas = frase["forma"].tolist()
        lemas = frase["lema"].tolist()
        idx = frase["i"].tolist()
        pos = frase["posicao"].tolist()
        uid = frase["unidade"].iloc[0]
        j, n = 0, len(formas)
        while j < n:
            for chave, categoria, termo in entradas:
                tam = len(chave)
                if j + tam > n:
                    continue
                if all(formas[j + k] == chave[k] or lemas[j + k] == chave[k] for k in range(tam)) and \
                   all(idx[j + k + 1] - idx[j + k] == 1 for k in range(tam - 1)):
                    achados.append({"unidade": uid, "posicao": pos[j], "categoria": categoria,
                                    "termo": termo, "forma": " ".join(formas[j:j + tam]), "n_tokens": tam})
                    j += tam
                    break
            else:
                j += 1
    return pd.DataFrame(achados, columns=["unidade", "posicao", "categoria", "termo", "forma", "n_tokens"])


def taxas_bootstrap(contagens, tamanhos, semente):
    """Bootstrap por unidades: reamostra unidades com reposição e recalcula a taxa por
    1.000 tokens. Respeita a dependência entre palavras de uma mesma unidade.
    contagens: matriz (rótulos x unidades). Devolve (rótulos x reamostras)."""
    rng = np.random.default_rng(semente)
    n_u = len(tamanhos)
    idx = rng.integers(0, n_u, size=(N_BOOTSTRAP, n_u))
    return contagens[:, idx].sum(axis=2) / tamanhos[idx].sum(axis=1) * 1000


def tabela_categorias(achados, tokens, categorias, semente):
    """Ocorrências por categoria, taxa por 1.000 tokens com IC por bootstrap,
    participação no total do codebook e alcance."""
    unidades = sorted(tokens["unidade"].unique())
    tamanhos = tokens.groupby("unidade").size().reindex(unidades).to_numpy()
    if achados.empty:
        matriz = pd.DataFrame(0, index=categorias, columns=unidades)
    else:
        matriz = pd.crosstab(achados["categoria"], achados["unidade"]).reindex(
            index=categorias, columns=unidades, fill_value=0)
    reamostras = taxas_bootstrap(matriz.to_numpy(), tamanhos, semente)
    alfa = (1 - NIVEL_CONFIANCA) / 2
    ocorr = matriz.sum(axis=1)
    tab = pd.DataFrame({
        "ocorrencias": ocorr,
        "por_1000_tokens": ocorr / len(tokens) * 1000,
        "ic_inferior": np.quantile(reamostras, alfa, axis=1),
        "ic_superior": np.quantile(reamostras, 1 - alfa, axis=1),
        "pct_do_codebook": ocorr / ocorr.sum() * 100 if ocorr.sum() else np.nan,
        "unidades_com_ocorrencia": (matriz > 0).sum(axis=1),
    })
    tab.index.name = "categoria"
    return tab, reamostras


def tabela_termos_codebook(achados, entradas, n_tokens):
    """Contribuição de cada termo do codebook, incluindo os termos sem ocorrência."""
    linhas = []
    for _, categoria, termo in sorted(entradas, key=lambda e: (list(CATEGORIAS).index(e[1]), e[2])):
        sub = achados[(achados["categoria"] == categoria) & (achados["termo"] == termo)]
        linhas.append({
            "categoria": categoria,
            "termo": termo,
            "ocorrencias": len(sub),
            "por_1000_tokens": len(sub) / n_tokens * 1000,
            "formas_observadas": "; ".join(f"{f} ({n})" for f, n in sub["forma"].value_counts().items()),
        })
    tab = pd.DataFrame(linhas)
    total_cat = tab.groupby("categoria")["ocorrencias"].transform("sum")
    tab["pct_da_categoria"] = np.where(total_cat > 0, tab["ocorrencias"] / total_cat.where(total_cat > 0, 1) * 100, np.nan)
    return tab


def robustez_termo_dominante(termos, n_tokens):
    """Teste de robustez: quanto cada categoria depende do seu termo mais frequente."""
    linhas = []
    for categoria in CATEGORIAS:
        sub = termos[termos["categoria"] == categoria].sort_values(
            ["ocorrencias", "termo"], ascending=[False, True])
        total = sub["ocorrencias"].sum()
        dominante = sub.iloc[0]
        linhas.append({
            "categoria": categoria,
            "termo_dominante": dominante["termo"] if total else "—",
            "ocorrencias_do_termo": dominante["ocorrencias"] if total else 0,
            "participacao_do_termo_pct": dominante["ocorrencias"] / total * 100 if total else np.nan,
            "por_1000_com_o_termo": total / n_tokens * 1000,
            "por_1000_sem_o_termo": (total - dominante["ocorrencias"]) / n_tokens * 1000,
        })
    tab = pd.DataFrame(linhas).set_index("categoria")
    tab["alerta"] = np.where(tab["participacao_do_termo_pct"] > 50,
                             "mais da metade da categoria vem de um único termo", "")
    return tab


MODAIS = ["must", "shall", "should", "will", "would", "may", "might", "can", "could"]


def tabela_modais(tokens):
    """Verbos modais (etiqueta gramatical MD), por lema, por 1.000 tokens.
    Ordem fixa; modais fora da lista padrão, se houver, entram ao final."""
    modais = tokens[tokens["etiqueta"] == "MD"]["lema"].value_counts()
    ordem = MODAIS + sorted(set(modais.index) - set(MODAIS))
    modais = modais.reindex(ordem, fill_value=0)
    return pd.DataFrame({"ocorrencias": modais, "por_1000_tokens": modais / len(tokens) * 1000}).rename_axis("modal")


def grafico_dispersao_posicional(ax, posicoes_por_linha, rotulos, inicios_unidades, cor):
    """Gráfico de dispersão posicional: cada traço é uma ocorrência, na posição
    relativa em que aparece no documento (0 = início; 1 = fim)."""
    if len(inicios_unidades) <= 120:
        for x in inicios_unidades:
            ax.axvline(x, color=COR_GRADE, lw=0.5, zorder=0)
    for y, posicoes in enumerate(posicoes_por_linha):
        ax.vlines(posicoes, y - 0.36, y + 0.36, color=cor, lw=0.9)
    ax.set_yticks(range(len(rotulos)), labels=rotulos)
    ax.set_ylim(len(rotulos) - 0.5, -0.5)
    ax.set_xlim(0, 1)
    ax.grid(False)
    ax.xaxis.set_major_formatter(FMT_BR)
    ax.set_xlabel("Posição relativa no documento (0 = início; 1 = fim)")


def versoes_ambiente(nlp):
    return {
        "python": platform.python_version(),
        "spacy": spacy.__version__,
        "modelo_spacy": f"{MODELO_SPACY} {nlp.meta.get('version', '?')}",
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
    }


## 9. Carregamento e verificação do insumo

O JSON da skill 01 é lido em modo somente leitura. A execução é interrompida se: o arquivo não existir; não tiver sido produzido pela skill 01 para o corpus *Planos Nacionais de IA*; os identificadores não forem sequenciais; houver unidade vazia ou de tipo desconhecido; o número de unidades (total e por tipo) divergir do registrado na extração; faltar o título; ou o idioma não for `en`. O SHA-256 do JSON é exibido. Em seguida, os filtros de recorte da configuração são aplicados.

In [ ]:
documento = carregar_extracao(PASTA / ARQUIVO_JSON)

if documento["idioma"] != IDIOMA_ESPERADO:
    raise ValueError(
        f"Idioma declarado no JSON: {documento['idioma']!r}; esperado: {IDIOMA_ESPERADO!r}. "
        "Todos os planos da pasta devem estar em inglês (skill, seção 1.1)."
    )

selecionadas, retiradas = selecionar_unidades(documento)
unidades = [(u["id"], u["texto"]) for u in selecionadas]       # (posição, texto analisado)
TIPO_UNIDADE = {u["id"]: u["tipo"] for u in selecionadas}

tipos_json = pd.Series([u["tipo"] for u in documento["unidades"]]).value_counts()
print(f"Título: {documento['titulo']}")
print(f"Subtítulo: {documento['subtitulo'] or '— (não há)'}")
print(f"Arquivo: {documento['arquivo']}")
print(f"SHA-256: {documento['sha256']}")
print(f"Idioma: {documento['idioma']}")
print(f"Unidades no JSON: {len(documento['unidades'])} ("
      + "; ".join(f"{ROTULO_TIPO[t]}: {int(tipos_json.get(t, 0))}" for t in ("heading", "paragraph", "list_item")) + ")")
print(f"Título incluído nas contagens: {'sim (unidade 0)' if INCLUIR_TITULO_NA_CONTAGEM else 'não'}")
print(f"Unidades analisadas: {len(unidades)}; retiradas pelos filtros: {len(retiradas)} (detalhes na seção 10)")


## 10. Pré-processamento e perfil do documento

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
nlp = ajustar_tokenizador(spacy.load(MODELO_SPACY))
STOPWORDS = montar_stopwords(nlp)
tokens = processar_documento(unidades, nlp, STOPWORDS)
tokens_retirados = processar_documento([(u["id"], u["texto"]) for u in retiradas], nlp, STOPWORDS)
N_TOKENS = len(tokens)
N_UNIDADES = tokens["unidade"].nunique()

display(Markdown("**Filtros de recorte:** unidades e tokens retirados por cada filtro da configuração"))
filtros = tabela_filtros(retiradas, tokens_retirados)
exibir(filtros, casas=0)
salvar_tabela(filtros, "00_filtros_recorte")

display(Markdown("**Perfil do documento**"))
perfil = perfil_documento(tokens, TIPO_UNIDADE, SIGLA).to_frame()
exibir(perfil)
salvar_tabela(perfil, "00_perfil_documento")
salvar_tabela(tokens, "00_tokens_processados", index=False)

display(Markdown("**Ajustes de tokenização:** formas do texto afetadas"))
ajustes = tabela_ajustes_tokenizacao(unidades)
if ajustes.empty:
    print("Nenhuma forma do texto é afetada pelos ajustes de tokenização.")
else:
    exibir(ajustes.set_index(["ajuste", "forma_no_texto"]), casas=0)
salvar_tabela(ajustes, "00_ajustes_tokenizacao", index=False)

display(Markdown("**Lemas de classes de conteúdo descartados pela lista de *stopwords***"))
descartados = tabela_stopwords_descartadas(tokens, STOPWORDS)
exibir(descartados, casas=0)
salvar_tabela(descartados, "00_lemas_descartados_stopwords")

unidades_rep, frases_rep = exame_de_repeticoes(unidades, tokens)
print(f"Exame de inflação: {len(unidades_rep)} unidade(s) com texto idêntico; {len(frases_rep)} frase(s) "
      f"idêntica(s) repetida(s) com {TAMANHO_MINIMO_FRASE_REPETIDA} tokens ou mais.")
if len(unidades_rep):
    exibir(unidades_rep)
if len(frases_rep):
    exibir(frases_rep)
print(f"Tratamentos de inflação aplicados (UNIDADES_EXCLUIDAS): {UNIDADES_EXCLUIDAS or 'nenhum'}")
print(f"Lemas de conteúdo descartados pela lista de stopwords: {len(descartados)} "
      f"({int(descartados['frequencia'].sum()) if len(descartados) else 0} ocorrências)")
print(f"Stopwords aplicadas: {len(STOPWORDS)} (lista do spaCy, {len(STOPWORDS_ADICIONAIS)} adicionada(s), "
      f"{len(STOPWORDS_MANTIDAS)} mantida(s))")
print(f"Correções de lema aplicadas: {CORRECOES_DE_LEMA}")
print("Ambiente: " + "; ".join(f"{k} {v}" for k, v in versoes_ambiente(nlp).items()))


**Método e escolhas.** Cada unidade é processada com o modelo `en_core_web_sm` do spaCy. Palavras compostas com hífen (*AI-driven*, *open-source*) são reunidas em um só token. **Token** é toda palavra formada por letras (com hífen interno ou sigla com pontos); números (*2027*, *70%*), pontuação e símbolos não são tokens. O total de tokens é o **denominador** de todas as taxas por 1.000 tokens. **Tokens de conteúdo** são substantivos, nomes próprios, verbos, adjetivos e advérbios fora da lista de *stopwords* do spaCy, verificada pelo **lema**, para que todas as flexões de uma palavra recebam o mesmo tratamento. Conforme o padrão da skill 02 (seção 3.1), entram o título (unidade 0) e os títulos internos, e nenhuma categoria ou unidade é retirada. A primeira tabela registra quantas unidades e quantos tokens cada filtro retirou. A tabela de tokens processados (`00_tokens_processados.csv`, exportada para `resultados/` apenas por comando do pesquisador) permite conferir, token a token, a forma, o lema, a classe gramatical e a decisão de inclusão.

**Ajustes de tokenização.** Dois ajustes, visíveis na configuração e reversíveis, impedem que palavras escritas no plano fiquem fora da contagem por um efeito do tokenizador do spaCy, que mantém certos símbolos colados à palavra. Com o símbolo colado, a palavra inteira deixa de ser reconhecida como token.
1. `SEPARAR_SIMBOLO_FINAL`: o "+" colado à palavra (*AI+*, *Intelligence+*) e o hífen suspenso (*intelligentization-* em "intelligentization- and innovation-oriented") são separados. A palavra é contada, e o símbolo não, como define a skill 02 (seção 3.3).
2. `SIGLAS_COM_E_COMERCIAL`: as siglas com "&" (*R&D*, *S&T*) contam como um token, como as siglas com pontos (*U.S.*). Este é um acréscimo à definição de token da skill 02, que não prevê siglas com "&".

Neste plano, os ajustes afetam **19 ocorrências** (tabela acima): *AI+* (9), *Intelligence+* (1) e *intelligence+* (1), contadas nos lemas *ai* e *intelligence*; *intelligentization-* (1) e *medium-* (1); *R&D* (3) e *S&T* (3), contadas como os lemas *r&d* e *s&t*. Sem os ajustes, essas 19 ocorrências ficariam fora de todas as contagens, inclusive do total de tokens. As 9 ocorrências de *AI+* estão entre as 75 do lema *ai*.

**Exame de inflação** (skill 02, seção 3.5). Nenhuma unidade tem texto idêntico ao de outra, e nenhuma frase com 8 tokens ou mais (`TAMANHO_MINIMO_FRASE_REPETIDA`) se repete. Seis títulos internos começam com *“AI+”* (*“AI+” Science and Technology*, *“AI+” Industry Development* etc.), mas cada um é distinto e aparece uma única vez no texto: é frequência real, e não artefato, e por isso permanece na análise (skill 02, seção 3.5). Esses seis títulos respondem por 6 das 75 ocorrências de *ai*. Nenhum tratamento de inflação foi aplicado (`UNIDADES_EXCLUIDAS = {}`).

**Como ler.** O perfil descreve a extensão do documento e de seu vocabulário. Ele serve de referência para dimensionar as frequências apresentadas a seguir. A tabela de lemas descartados mostra o que a lista de *stopwords* retirou da contagem de conteúdo, para que o pesquisador decida se algum deve voltar (`STOPWORDS_MANTIDAS`).

**Interpretação.** *Observação.* Foram analisadas 65 unidades (o título, 35 títulos internos e 29 parágrafos; o plano não tem itens de lista), com 2.874 tokens. Os tokens de conteúdo são 1.972 (68,62% do total) e se distribuem em 728 lemas distintos, dos quais 419 (57,6%) ocorrem uma única vez. A mediana de 11 tokens por unidade, muito abaixo da média de 44,22, decorre do grande número de títulos internos curtos (seção 11). Nenhum filtro retirou unidades, e o exame de inflação não encontrou unidades nem frases repetidas. A lista de *stopwords* retirou da contagem de conteúdo 14 lemas, com 44 ocorrências. Os mais frequentes são *such* (15, todas na locução *such as*), *more* (8) e *give* (4).

*Inferência:* a ausência de repetições indica que as frequências das seções seguintes não estão infladas por trechos duplicados ou formulares. A proporção de lemas que ocorrem uma única vez (57,6%) sugere um vocabulário amplo em relação à extensão do texto, mas essa medida depende da extensão e não serve, sozinha, como indicador de riqueza vocabular (skill 02, seção 3.4).

**Fragilidades e ajustes possíveis.** O lematizador e o etiquetador do spaCy cometem erros, sobretudo em siglas e termos técnicos. Por isso, as tabelas trazem as formas observadas de cada lema. Correções conhecidas ficam explícitas em `CORRECOES_DE_LEMA`. A lista de *stopwords* do spaCy inclui palavras como *will*, *shall*, *may* e *must*, que ficam fora da contagem de conteúdo. Formas com algarismos não são tokens (*6G* é dividido pelo spaCy em *6*, que não é token, e *G*, token de uma letra que conta no total de tokens, mas não como conteúdo). 

## 11. Extensão das unidades textuais

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
tam_unidades = tokens.groupby("unidade").size().reindex([u for u, _ in unidades], fill_value=0)
estrutural = np.array([TIPO_UNIDADE[u] in ("title", "heading") for u in tam_unidades.index])

fig, ax = plt.subplots(figsize=(7.5, 3.2))
x = np.arange(len(tam_unidades))
ax.bar(x, tam_unidades.to_numpy(), width=0.8, color=np.where(estrutural, COR_ESTRUTURA, COR))
ax.axhline(tam_unidades.mean(), color=COR_TINTA_2, lw=0.8)
# Rótulo da média fora da área das barras, à direita do eixo, para não ficar encoberto.
ax.text(1.01, tam_unidades.mean(), f"média =\n{num(tam_unidades.mean(), 1)}", transform=ax.get_yaxis_transform(),
        ha="left", va="center", fontsize=9, color=COR_TINTA_2)
passo = max(1, int(np.ceil(len(x) / 20)))
ax.set_xticks(x[::passo], labels=[str(u) for u in tam_unidades.index[::passo]])
ax.set_xlim(-0.6, len(x) - 0.4)
ax.set_xlabel("Unidade (id; 0 = título), na ordem do documento")
ax.set_ylabel("Tokens (palavras)")
ax.set_title(f"{SIGLA} · extensão das unidades textuais", loc="left")
ax.grid(axis="x", visible=False)
ax.yaxis.set_major_formatter(FMT_BR)
if estrutural.any() and (~estrutural).any():
    ax.legend(handles=[Patch(color=COR, label="parágrafos e itens de lista"),
                       Patch(color=COR_ESTRUTURA, label="título e títulos internos")],
              loc="upper center", bbox_to_anchor=(0.5, -0.26), ncols=2, handlelength=1.2)
salvar_figura(fig, "01_extensao_unidades")
plt.show()

salvar_tabela(tam_unidades.rename("tokens").to_frame().assign(
    tipo=[TIPO_UNIDADE[u] for u in tam_unidades.index]), "01_extensao_unidades")
maior, menor = tam_unidades.idxmax(), tam_unidades.idxmin()
print(f"Síntese factual: {len(tam_unidades)} unidades; média de {num(tam_unidades.mean(), 1)} tokens "
      f"(desvio-padrão {num(tam_unidades.std(), 1)}). Maior unidade: id {maior} ({tam_unidades[maior]} tokens); "
      f"menor: id {menor} ({tam_unidades[menor]} tokens).")
for nome, mascara in (("título e títulos internos", estrutural), ("parágrafos e itens de lista", ~estrutural)):
    if mascara.any():
        grupo = tam_unidades[mascara]
        print(f"  {nome}: {len(grupo)} unidades, {grupo.sum()} tokens ({num(grupo.sum() / N_TOKENS * 100, 1)}% do total); "
              f"média de {num(grupo.mean(), 1)} tokens (mínimo {grupo.min()}; máximo {grupo.max()}).")


**Método e escolhas.** Número de tokens (palavras) de cada unidade, na ordem original do documento. A linha horizontal marca a média de todas as unidades. Nenhuma unidade é omitida. Como o título (unidade 0) e os títulos internos entram na análise por padrão, as barras desses elementos aparecem em cinza, e as dos parágrafos e itens de lista, na cor do plano. A síntese factual informa a extensão de cada grupo separadamente.

**Como ler.** Cada barra é uma unidade: o título, um título interno, um parágrafo ou um item de lista. Barras altas indicam unidades extensas; a sequência mostra como o texto se organiza (por exemplo, títulos curtos que antecedem parágrafos longos).

**Interpretação.** *Observação.* As 65 unidades têm, em média, 44,2 tokens (desvio-padrão de 44,8), mas os dois grupos são muito diferentes. O título e os 35 títulos internos (36 unidades) somam 223 tokens (7,8% do total), com média de 6,2 tokens (de 2 a 14). Os 29 parágrafos somam 2.651 tokens (92,2%), com média de 91,4 tokens (de 60 a 135). A média geral, por isso, não descreve bem nenhum dos dois grupos. Os parágrafos mais longos são os de id 4 (135 tokens) e id 3 (134), os dois que seguem o título *Overall Requirements*, e depois os de id 36 (127) e id 33 (126). O mais curto é o de id 40 (60 tokens). Com exceção dos parágrafos de id 1 (logo após o título do plano) e id 4 (segundo parágrafo sob *Overall Requirements*), cada parágrafo vem logo depois de um ou mais títulos internos.

*Inferência:* a alternância regular entre títulos curtos e parágrafos de extensão relativamente homogênea sugere um texto organizado em itens, cada um desenvolvido em um único parágrafo. A extensão de um parágrafo, por si só, não indica a importância do tema que ele trata.

**Fragilidades e ajustes possíveis.** A segmentação em unidades é a da skill 01, que segue a estrutura do original. A média de todas as unidades mistura títulos e parágrafos, que têm extensões de natureza diferente; por isso, ela deve ser lida junto com as médias por grupo da síntese factual. Unidades muito curtas, como os títulos internos, tornam instáveis as medidas por unidade (alcance e dispersão), o que deve ser lembrado na leitura das seções seguintes. 

## 12. Lemas de conteúdo mais frequentes

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
freq = tabela_frequencias(tokens)
salvar_tabela(freq, "02_frequencias_lemas_completa")
relatar_empates(freq, "frequencia", TOP_N_LEMAS, "lema")
top = freq.head(TOP_N_LEMAS)

fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(top) + 1.3))
y = np.arange(len(top))
ax.barh(y, top["frequencia"], height=0.7, color=COR)
ax.set_yticks(y, labels=top.index)
ax.invert_yaxis()
xmax = top["frequencia"].max()
for yi, (_, linha) in zip(y, top.iterrows()):
    ax.text(linha["frequencia"] + xmax * 0.015, yi,
            f"{num(linha['por_1000_tokens'])} por 1.000 · {int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
            va="center", fontsize=9, color=COR_TINTA_2)
ax.set_xlim(0, xmax * 1.55)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.xaxis.set_major_formatter(FMT_BR)
ax.set_xlabel("Frequência absoluta (ocorrências)")
ax.set_title(f"{SIGLA} · {len(top)} lemas de conteúdo mais frequentes", loc="left")
ax.grid(axis="y", visible=False)
salvar_figura(fig, "02_lemas_mais_frequentes")
plt.show()

exibir(top[["frequencia", "por_1000_tokens", "unidades_com_ocorrencia", "pct_unidades", "DP_norm", "formas_observadas"]])
n_conteudo = int(tokens["conteudo"].sum())
print(f"Síntese factual: os {len(top)} lemas acima somam {int(top['frequencia'].sum())} ocorrências, "
      f"{num(top['frequencia'].sum() / n_conteudo * 100, 1)}% dos {n_conteudo} tokens de conteúdo. "
      f"O documento tem {len(freq)} lemas de conteúdo distintos (tabela completa exportada).")


**Método e escolhas.** Frequência absoluta dos *N* lemas de conteúdo mais frequentes (*N* = `TOP_N_LEMAS`). O rótulo de cada barra traz a frequência por 1.000 tokens e o **alcance** (em quantas unidades o lema aparece). A tabela acrescenta o **DP normalizado** (dispersão) e as formas observadas. Ordenação: frequência decrescente; empates resolvidos por ordem alfabética e informados acima do gráfico quando afetam o corte. A tabela completa de todos os lemas (`02_frequencias_lemas_completa.csv`) é exportada para `resultados/` apenas por comando do pesquisador.

**Como ler.** Barras mais longas indicam lemas mais frequentes. Um lema frequente presente em muitas unidades é estruturante no texto; um lema frequente presente em poucas unidades é localizado. A distinção aparece no rótulo (alcance) e na coluna DP_norm da tabela (0 = espalhado; 1 = concentrado).

**Interpretação.** *Observação.* *ai* é o lema mais frequente, com 75 ocorrências (26,10 por 1.000 tokens), presentes em 34 das 65 unidades (52,3%). É quase o dobro do segundo lema, *new* (41 ocorrências; 14,27 por 1.000). Das 75 ocorrências de *ai*, 9 vêm da forma *AI+* (seção 10) e 8 estão em títulos internos. A forma por extenso, *artificial intelligence*, aparece só 2 vezes (seção 14). Entre os 25 lemas do recorte, seis são verbos, quase sempre na forma básica: *promote* (35), *strengthen* (29), *accelerate* (23), *support* (16), *explore* (11) e *build* (10). Outros dois verbos, *create* e *improve*, empatam com 10 ocorrências logo fora do corte. Os adjetivos *intelligent* (32) e *smart* (15) somam 47 ocorrências. Depois de *ai*, os substantivos mais frequentes são *development* (37), *application* (32), *model* (26) e *service* (26). Os 25 lemas somam 557 ocorrências, 28,2% dos 1.972 tokens de conteúdo. Alguns lemas do recorte aparecem em poucas unidades: *compute* (10 ocorrências em 3 unidades; DP 0,93), *research* (12 em 7; DP 0,86) e *data* (13 em 7; DP 0,80).

*Inferência:* o peso dos verbos de ação no topo da lista é compatível com um texto de caráter diretivo, que enuncia o que deve ser feito. O par *intelligent*/*smart* sugere que qualificar objetos, serviços e setores como inteligentes é um traço recorrente do vocabulário do plano. As duas leituras dependem do contexto das ocorrências e não decorrem só da frequência.

**Fragilidades e ajustes possíveis.** Frequência não equivale a importância política: um termo pode ser frequente por razões de estilo ou por repetição formular. Lemas genéricos, como o próprio *ai*, tendem a ocupar o topo em planos sobre IA. Isso é um resultado, e não um erro, e por isso esses lemas não são removidos sem decisão do pesquisador. *AI* e *artificial intelligence* não são unificados, porque isso alteraria o texto. Se o pesquisador desejar, a unificação pode ser feita por regra explícita na configuração. *model* reúne sentidos distintos: modelos de IA (*model training*, *model architecture*) e modelos de organização ou de negócio (*new models*, *service models*, *commercial models*). *compute* reúne 9 ocorrências da forma *compute* e 1 de *computing*, que o spaCy classificou como verbo; as outras 2 ocorrências de *computing* ficaram no lema *computing*. Essa divergência não pode ser corrigida em `CORRECOES_DE_LEMA`, que atua sobre o lema e afetaria também *compute*. As 9 ocorrências de *AI+* estão contadas em *ai*. Se o pesquisador preferir tratá-las à parte, o ajuste `SEPARAR_SIMBOLO_FINAL` pode ser revisto.

## 13. Distribuição dos lemas ao longo do documento

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
lemas_disp = freq.head(TOP_N_DISPERSAO).index.tolist()
inicios = (tokens.groupby("unidade")["posicao"].min() / N_TOKENS).to_numpy()
posicoes = [tokens.loc[tokens["conteudo"] & (tokens["lema"] == lema), "posicao"].to_numpy() / N_TOKENS
            for lema in lemas_disp]
rotulos = [f"{lema}  (DP = {num(freq.loc[lema, 'DP_norm'])})" for lema in lemas_disp]

fig, ax = plt.subplots(figsize=(7.5, 0.33 * len(lemas_disp) + 1.3))
grafico_dispersao_posicional(ax, posicoes, rotulos, inicios, COR)
ax.set_title(f"{SIGLA} · posição das ocorrências dos {len(lemas_disp)} lemas mais frequentes", loc="left")
salvar_figura(fig, "03_dispersao_lemas")
plt.show()

exibir(freq.head(TOP_N_DISPERSAO)[["frequencia", "unidades_com_ocorrencia", "DP_norm"]].sort_values("DP_norm"))


**Método e escolhas.** Cada traço vertical é uma ocorrência do lema, posicionada pela sua posição relativa no documento (posição do token ÷ total de tokens). As linhas verticais claras marcam o início de cada unidade. O DP normalizado (Gries, 2008; Lijffijt & Gries, 2012) resume a distribuição: 0 indica ocorrências proporcionais ao tamanho das unidades; 1, concentração máxima.

**Como ler.** Linhas com traços espalhados indicam termos presentes em todo o texto; traços agrupados indicam termos concentrados em uma parte (por exemplo, em uma seção específica). A tabela ordena os lemas do mais espalhado ao mais concentrado.

**Interpretação.** *Observação.* *ai* é o lema mais bem distribuído do recorte (DP = 0,29), seguido de *promote* (0,31) e *development* (0,37). No gráfico, *ai* aparece do início ao fim do documento, com maior densidade no trecho aberto pelo título *“AI+” Global Cooperation*. Esse trecho vai da posição 0,63 à 0,70 e reúne cerca de 6% dos tokens, mas concentra 12 das 75 ocorrências de *ai*. Nele está o parágrafo com a maior taxa do lema (id 43: 6 ocorrências em 84 tokens). Os lemas mais concentrados do recorte são *governance* (0,73), *capability* (0,72) e *service* (0,69).

- *governance*: 12 das 15 ocorrências estão entre as posições 0,52 e 0,69, trecho que vai do título *“AI+” Governance Capacity* até o fim da parte aberta por *“AI+” Global Cooperation*.
- *intelligent*: tem dois agrupamentos. São 12 ocorrências entre 0,34 e 0,42, em torno de *“AI+” Consumption Quality Improvement*, e 6 entre 0,75 e 0,79, sob *Strengthen the coordination of intelligent compute.*
- *service*: 11 das 26 ocorrências estão entre 0,32 e 0,38, e 7, entre 0,78 e 0,82.
- *strengthen*: é mais frequente no fim do texto. São 16 das 29 ocorrências a partir da posição 0,70, onde começa *Strengthening Basic Support Capabilities*, trecho que reúne cerca de 30% dos tokens. Cinco delas estão em títulos internos que começam com *Strengthen* ou *Strengthening*.
- *new*: concentra-se na primeira metade do texto, com 33 das 41 ocorrências antes da posição 0,5; 7 delas estão só no parágrafo de id 3.

*Inferência:* os lemas mais espalhados (*ai*, *promote*, *development*) funcionam como vocabulário transversal do plano. Os mais concentrados acompanham a divisão do texto em partes, o que indica que temas como governança, serviços e capacidade computacional são tratados em trechos específicos, e não ao longo de todo o documento.

**Fragilidades e ajustes possíveis.** Em documentos curtos ou com poucas unidades, o DP é sensível à segmentação: uma única unidade longa pode concentrar ocorrências por ser longa, e não por ênfase. Neste plano, as unidades curtas dos títulos internos se alternam com parágrafos longos; um lema ausente dos títulos tem, por isso, DP maior do que teria se só os parágrafos fossem considerados. O gráfico mostra posições, não intensidade. 

## 14. Expressões recorrentes (bigramas)

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
bigramas = tabela_bigramas(tokens)
salvar_tabela(bigramas, "04_bigramas_completa")
elegiveis = bigramas[bigramas["frequencia"] >= FREQ_MINIMA_BIGRAMA]
relatar_empates(elegiveis, "frequencia", TOP_N_BIGRAMAS, "bigrama")
topb = elegiveis.head(TOP_N_BIGRAMAS)

if topb.empty:
    print(f"Nenhum bigrama atinge a frequência mínima de {FREQ_MINIMA_BIGRAMA}; gráfico não gerado.")
else:
    fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(topb) + 1.3))
    y = np.arange(len(topb))
    ax.barh(y, topb["frequencia"], height=0.7, color=COR)
    ax.set_yticks(y, labels=topb["forma_principal"])
    ax.invert_yaxis()
    xmax = topb["frequencia"].max()
    for yi, (_, linha) in zip(y, topb.iterrows()):
        ax.text(linha["frequencia"] + xmax * 0.015, yi,
                f"{int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
                va="center", fontsize=9, color=COR_TINTA_2)
    ax.set_xlim(0, xmax * 1.25)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.xaxis.set_major_formatter(FMT_BR)
    ax.set_xlabel("Frequência absoluta (ocorrências)")
    ax.set_title(f"{SIGLA} · {len(topb)} expressões de duas palavras mais frequentes", loc="left")
    ax.grid(axis="y", visible=False)
    salvar_figura(fig, "04_bigramas")
    plt.show()
    exibir(topb)
print(f"Síntese factual: {len(bigramas)} bigramas distintos; {len(elegiveis)} com frequência >= {FREQ_MINIMA_BIGRAMA}.")


**Método e escolhas.** Pares de tokens de conteúdo **adjacentes no texto original**, na mesma frase e sem pontuação entre eles. Pares agrupados pelo lema (ex.: *new model* reúne *new models* e *new model*). O rótulo mostra a forma escrita mais frequente. Só entram no gráfico os bigramas com frequência mínima `FREQ_MINIMA_BIGRAMA`. Como a adjacência é verificada no texto original, a remoção de *stopwords* não cria pares artificiais (em *factors of production*, não se forma *factors production*). Pela mesma razão, o "+" separado de *AI+* impede pares como *ai science* nos títulos dos eixos.

**Como ler.** As barras mostram as expressões que o texto mais repete, frequentemente nomes de conceitos. O rótulo indica em quantas unidades a expressão aparece.

**Interpretação.** *Observação.* Há 797 bigramas distintos: 87 ocorrem duas vezes ou mais, e 710, uma única vez. O mais frequente é *ai application* (7 ocorrências, em 4 unidades). Quatro dos cinco seguintes são fragmentos de uma mesma expressão, *new generation smart terminal devices*, que o método de bigramas divide: *smart terminal* (6), *terminal devices* (6), *generation smart* (5) e *new generation* (5). Vêm depois *new model(s)* (5, em 5 unidades) e, com 4 ocorrências cada, *explore new*, *human-computer coordination*, *intelligent agents*, *intelligent compute* e *vigorously develop*. Com 3 ocorrências aparecem, entre outros, *ai governance*, *global ai*, *smart society* e *intelligent economy* (este último com as 3 ocorrências numa só unidade). O 20º lugar do gráfico (*agricultural machinery*, 2) é apenas o primeiro, em ordem alfabética, de 68 bigramas empatados com 2 ocorrências. Entre eles está *artificial intelligence*, que aparece só no título e no primeiro parágrafo.

*Inferência:* as expressões mais repetidas nomeiam sobretudo tecnologias e instrumentos de aplicação (dispositivos terminais, agentes inteligentes, capacidade computacional, aplicações de IA) e modos de ação (*explore new*, *vigorously develop*). As frequências, porém, são baixas (no máximo 7), e diferenças de uma ou duas ocorrências entre bigramas não devem ser lidas como diferenças de ênfase.

**Fragilidades e ajustes possíveis.** Expressões com três ou mais palavras aparecem fragmentadas em bigramas sobrepostos (*new generation smart terminal devices* gera *new generation*, *generation smart*, *smart terminal* e *terminal device*). Expressões interrompidas por *stopwords* (*factors of production*) não são captadas como bigramas. Em textos curtos, poucas expressões atingem a frequência mínima. Como 68 bigramas empatam com 2 ocorrências, a última posição do gráfico depende só do critério de desempate (ordem alfabética). A tabela completa (`04_bigramas_completa.csv`) traz todos eles.

## 15. Lemas de conteúdo mais frequentes: recorte ampliado para 30

*Visualização construída por comando do pesquisador (2026-10-05): ampliação do recorte da seção 12 de 25 para 30 lemas. A seção 12 foi mantida sem nenhuma alteração, para que a versão original continue reproduzível exatamente como estava.*

In [ ]:
# Parâmetro desta visualização, definido por comando do pesquisador (2026-10-05): recorte ampliado de
# TOP_N_LEMAS = 25 (seção 7) para 30 lemas. Fica nesta célula, e não na configuração, para que as células
# originais do notebook permaneçam inalteradas (exceção registrada no texto abaixo).
TOP_N_LEMAS_AMPLIADO = 30

# A tabela de frequências é a da seção 12 (freq: todos os lemas de conteúdo); só muda o recorte exibido.
relatar_empates(freq, "frequencia", TOP_N_LEMAS_AMPLIADO, "lema")
top_ampliado = freq.head(TOP_N_LEMAS_AMPLIADO)

fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(top_ampliado) + 1.3))
y = np.arange(len(top_ampliado))
ax.barh(y, top_ampliado["frequencia"], height=0.7, color=COR)
ax.set_yticks(y, labels=top_ampliado.index)
ax.invert_yaxis()
xmax = top_ampliado["frequencia"].max()
for yi, (_, linha) in zip(y, top_ampliado.iterrows()):
    ax.text(linha["frequencia"] + xmax * 0.015, yi,
            f"{num(linha['por_1000_tokens'])} por 1.000 · {int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
            va="center", fontsize=9, color=COR_TINTA_2)
ax.set_xlim(0, xmax * 1.55)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.xaxis.set_major_formatter(FMT_BR)
ax.set_xlabel("Frequência absoluta (ocorrências)")
ax.set_title(f"{SIGLA} · {len(top_ampliado)} lemas de conteúdo mais frequentes", loc="left")
ax.grid(axis="y", visible=False)
salvar_figura(fig, f"05_lemas_mais_frequentes_{TOP_N_LEMAS_AMPLIADO}")
plt.show()

exibir(top_ampliado[["frequencia", "por_1000_tokens", "unidades_com_ocorrencia", "pct_unidades", "DP_norm", "formas_observadas"]])
n_conteudo = int(tokens["conteudo"].sum())
acrescentados = top_ampliado.index[TOP_N_LEMAS:]
print(f"Síntese factual: os {len(top_ampliado)} lemas acima somam {int(top_ampliado['frequencia'].sum())} ocorrências, "
      f"{num(top_ampliado['frequencia'].sum() / n_conteudo * 100, 1)}% dos {n_conteudo} tokens de conteúdo. "
      f"Lemas acrescentados ao recorte da seção 12 ({TOP_N_LEMAS} lemas): "
      + ", ".join(f"{lema} ({int(freq.loc[lema, 'frequencia'])})" for lema in acrescentados) + ".")


**Método e escolhas.** Mesma medida e mesmo gráfico da seção 12: frequência absoluta dos *N* lemas de conteúdo mais frequentes, com a frequência por 1.000 tokens e o **alcance** no rótulo de cada barra, e o **DP normalizado** e as formas observadas na tabela. A única diferença é o recorte: *N* = `TOP_N_LEMAS_AMPLIADO` = 30, por comando do pesquisador, em vez de `TOP_N_LEMAS` = 25. A tabela de frequências é a mesma da seção 12 (`freq`, com todos os lemas de conteúdo); nenhum valor é recalculado de outra forma. Ordenação: frequência decrescente; empates resolvidos por ordem alfabética e informados acima do gráfico quando afetam o corte. O parâmetro `TOP_N_LEMAS_AMPLIADO` foi definido nesta seção, e não na célula de configuração (seção 7), para que as células originais do notebook permanecessem inalteradas e a versão de 25 lemas continuasse reproduzível. Essa exceção à centralização dos parâmetros (skill 02, seção 4.1) fica registrada aqui. Esta seção foi executada isoladamente, sem reexecutar as anteriores.

**Como ler.** Barras mais longas indicam lemas mais frequentes. Um lema frequente presente em muitas unidades é estruturante no texto; um lema frequente presente em poucas unidades é localizado. A distinção aparece no rótulo (alcance) e na coluna DP_norm da tabela (0 = espalhado; 1 = concentrado). Os 25 primeiros lemas são os mesmos da seção 12, na mesma ordem; a síntese factual lista os lemas acrescentados.

**Interpretação.** *Observação.* Com o recorte de 30, entram cinco lemas: *create* (10 ocorrências, em 10 unidades), *improve* (10, em 9), *consumption* (9, em 5), *encourage* (9, em 8) e *management* (9, em 7). Os 30 lemas somam 604 ocorrências, 30,6% dos 1.972 tokens de conteúdo; com 25 lemas, eram 557 ocorrências e 28,2%. Três dos cinco lemas acrescentados são verbos (*create*, *improve* e *encourage*). Com eles, o recorte passa a ter nove verbos: *promote*, *strengthen*, *accelerate*, *support*, *explore*, *build*, *create*, *improve* e *encourage*. Os três novos verbos aparecem, em geral, uma única vez em cada unidade (DP entre 0,72 e 0,78). Já *consumption* é o mais concentrado dos acrescentados (DP 0,92): 5 de suas 9 ocorrências estão no parágrafo de id 24, sob o título *Expand new service consumption scenarios.*

O corte cai num empate. *consumption*, *encourage* e *management* são os três primeiros, em ordem alfabética, de sete lemas com 9 ocorrências. Os outros quatro ficaram fora do gráfico só pelo critério de desempate: *national* (9 ocorrências, em 6 unidades), *open-source* (9, em 3), *security* (9, em 6) e *talent* (9, em 3). *open-source* e *talent* concentram 7 ocorrências cada num único parágrafo (ids 56 e 58).

*Inferência:* a ampliação reforça a leitura da seção 12 de que os verbos de ação ocupam parte importante do vocabulário mais frequente, o que é compatível com um texto de caráter diretivo. Os novos verbos se espalham pelo texto, enquanto *consumption* se liga a um trecho específico. Como as posições 28 a 30 dependem do desempate alfabético, a presença de *consumption*, *encourage* e *management* e a ausência de *national*, *open-source*, *security* e *talent* não devem ser lidas como diferença de ênfase.

**Fragilidades e ajustes possíveis.** Ampliar o recorte não altera nenhum valor: os lemas, as frequências e as medidas são os da tabela completa. As fragilidades da seção 12 continuam válidas: frequência não equivale a importância política; *model* reúne sentidos distintos; *compute* inclui 1 ocorrência de *computing*; e as 9 ocorrências de *AI+* estão contadas em *ai*. Com 30 lemas, o 30º lugar cai num empate de sete lemas com 9 ocorrências, dos quais só três entram no gráfico, por ordem alfabética. Um recorte de 27 lemas (que termina em *improve*, com 10 ocorrências) ou de 34 (que inclui todos os empatados com 9) evitaria que o desempate decidisse quais lemas aparecem. A escolha é do pesquisador.

## 16. Lemas de conteúdo mais frequentes: recorte ampliado para 34, sem empate no corte

*Visualização construída por comando do pesquisador (2026-10-05): ampliação do recorte para 34 lemas, para que o ponto de corte não caia num empate de frequência (o recorte de 30, da seção 15, deixava de fora, só pelo desempate alfabético, quatro lemas com a mesma frequência do 30º). As seções 12 e 15 foram mantidas sem nenhuma alteração.*

In [ ]:
# Parâmetro desta visualização, definido por comando do pesquisador (2026-10-05): recorte de 34 lemas,
# escolhido para que o ponto de corte não caia num empate de frequência. Fica nesta célula, e não na
# configuração, para que as células originais do notebook permaneçam inalteradas (exceção registrada abaixo).
TOP_N_LEMAS_SEM_EMPATE = 34

# A tabela de frequências é a da seção 12 (freq: todos os lemas de conteúdo); só muda o recorte exibido.
relatar_empates(freq, "frequencia", TOP_N_LEMAS_SEM_EMPATE, "lema")   # nada é impresso se não houver empate
ultimo = int(freq["frequencia"].iloc[TOP_N_LEMAS_SEM_EMPATE - 1])
seguinte = int(freq["frequencia"].iloc[TOP_N_LEMAS_SEM_EMPATE]) if len(freq) > TOP_N_LEMAS_SEM_EMPATE else None
print(f"Ponto de corte: o {TOP_N_LEMAS_SEM_EMPATE}º lema tem {ultimo} ocorrências; o {TOP_N_LEMAS_SEM_EMPATE + 1}º, "
      f"{seguinte}. " + ("Não há empate no corte." if seguinte is None or seguinte < ultimo else "ATENÇÃO: há empate no corte."))
top_sem_empate = freq.head(TOP_N_LEMAS_SEM_EMPATE)

fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(top_sem_empate) + 1.3))
y = np.arange(len(top_sem_empate))
ax.barh(y, top_sem_empate["frequencia"], height=0.7, color=COR)
ax.set_yticks(y, labels=top_sem_empate.index)
ax.invert_yaxis()
xmax = top_sem_empate["frequencia"].max()
for yi, (_, linha) in zip(y, top_sem_empate.iterrows()):
    ax.text(linha["frequencia"] + xmax * 0.015, yi,
            f"{num(linha['por_1000_tokens'])} por 1.000 · {int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
            va="center", fontsize=9, color=COR_TINTA_2)
ax.set_xlim(0, xmax * 1.55)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.xaxis.set_major_formatter(FMT_BR)
ax.set_xlabel("Frequência absoluta (ocorrências)")
ax.set_title(f"{SIGLA} · {len(top_sem_empate)} lemas de conteúdo mais frequentes", loc="left")
ax.grid(axis="y", visible=False)
salvar_figura(fig, f"06_lemas_mais_frequentes_{TOP_N_LEMAS_SEM_EMPATE}")
plt.show()

exibir(top_sem_empate[["frequencia", "por_1000_tokens", "unidades_com_ocorrencia", "pct_unidades", "DP_norm", "formas_observadas"]])
n_conteudo = int(tokens["conteudo"].sum())
print(f"Síntese factual: os {len(top_sem_empate)} lemas acima somam {int(top_sem_empate['frequencia'].sum())} ocorrências, "
      f"{num(top_sem_empate['frequencia'].sum() / n_conteudo * 100, 1)}% dos {n_conteudo} tokens de conteúdo. "
      f"Lemas acrescentados ao recorte da seção 15 ({TOP_N_LEMAS_AMPLIADO} lemas): "
      + ", ".join(f"{lema} ({int(freq.loc[lema, 'frequencia'])})" for lema in top_sem_empate.index[TOP_N_LEMAS_AMPLIADO:]) + ".")


**Método e escolhas.** Mesma medida e mesmo gráfico das seções 12 e 15: frequência absoluta dos *N* lemas de conteúdo mais frequentes, com a frequência por 1.000 tokens e o **alcance** no rótulo de cada barra, e o **DP normalizado** e as formas observadas na tabela. A única diferença é o recorte: *N* = `TOP_N_LEMAS_SEM_EMPATE` = 34, por comando do pesquisador. O valor foi escolhido para que o ponto de corte não caia num empate: a célula verifica e imprime a frequência do 34º e do 35º lema. Assim, nenhum lema com a mesma frequência do último exibido fica de fora pelo critério de desempate. A tabela de frequências é a mesma da seção 12 (`freq`, com todos os lemas de conteúdo); nenhum valor é recalculado de outra forma. Dentro do recorte, lemas com a mesma frequência continuam ordenados alfabeticamente. O parâmetro `TOP_N_LEMAS_SEM_EMPATE` foi definido nesta seção, e não na célula de configuração (seção 7), para que as células originais permanecessem inalteradas. Essa exceção à centralização dos parâmetros (skill 02, seção 4.1) fica registrada aqui. Esta seção foi executada isoladamente, sem reexecutar as anteriores.

**Como ler.** Barras mais longas indicam lemas mais frequentes. Um lema frequente presente em muitas unidades é estruturante no texto; um lema frequente presente em poucas unidades é localizado. A distinção aparece no rótulo (alcance) e na coluna DP_norm da tabela (0 = espalhado; 1 = concentrado). Os 30 primeiros lemas são os mesmos da seção 15, na mesma ordem; a síntese factual lista os lemas acrescentados.

**Interpretação.** *Observação.* Com 34 lemas, o corte não cai mais num empate. O 34º lema tem 9 ocorrências e o 35º tem 8, de modo que os sete lemas com 9 ocorrências aparecem todos, nas posições 28 a 34. Em relação à seção 15, entram quatro lemas, todos com 9 ocorrências:

- *national*: 6 unidades; DP 0,79;
- *open-source*: 3 unidades; DP 0,94;
- *security*: 6 unidades; DP 0,85;
- *talent*: 3 unidades; DP 0,93; formas *talent* (5) e *talents* (4).

Os 34 lemas somam 640 ocorrências, 32,5% dos 1.972 tokens de conteúdo; eram 604 (30,6%) com 30 lemas e 557 (28,2%) com 25. Os quatro acrescentados são três substantivos e um adjetivo, e o número de verbos do recorte continua em nove. Três dos quatro concentram a maior parte das ocorrências num único parágrafo:

- *open-source*: 7 de 9 no parágrafo de id 56, sob *Promote a thriving open-source ecosystem.*;
- *talent*: 7 de 9 no parágrafo de id 58, sob *Strengthen talent cadre construction.*;
- *security*: 4 de 9 no parágrafo de id 38, sob o título sobre *safety and security governance*.

*national* tem 3 de suas 9 ocorrências no parágrafo de id 52, sob *Strengthen the coordination of intelligent compute.*

*Inferência:* os lemas que completam o recorte sem empate são, em geral, concentrados (DP de 0,79 a 0,94). Sua presença no fim da lista indica temas tratados em trechos específicos do plano (ecossistema de código aberto, formação de talentos, segurança), e não um vocabulário que atravessa o documento. Entre os sete lemas com 9 ocorrências, a ordem é apenas alfabética e não indica diferença de ênfase.

**Fragilidades e ajustes possíveis.** Ampliar o recorte não altera nenhum valor: os lemas, as frequências e as medidas são os da tabela completa. O recorte sem empate resolve a arbitrariedade no **ponto de corte**, mas não a ordem **dentro** do recorte: lemas com a mesma frequência continuam em ordem alfabética, e essa ordem não indica diferença de ênfase. As fragilidades da seção 12 continuam válidas: frequência não equivale a importância política; *model* reúne sentidos distintos; *compute* inclui 1 ocorrência de *computing*; e as 9 ocorrências de *AI+* estão contadas em *ai*. *open-source* e *talent* dependem quase inteiramente de um único parágrafo (7 de 9 ocorrências cada). A presença deles no recorte é, portanto, sensível à segmentação e ao conteúdo dessas unidades, o que a coluna DP_norm (0,94 e 0,93) já indica.